# installs and imports

In [ ]:
pip install PyCO2SYS

In [ ]:
import PyCO2SYS as pyco2
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import gcsfs
fs = gcsfs.GCSFileSystem()

import smtplib

# perform carbonate system calculation

In [ ]:
# variables you want PyCO2SYS to calculate
# variables not listed here will not be kept so as to not overload the memory.
output_vars = ['dic','carbonate','bicarbonate','aqueous_CO2','pCO2','alkalinity','pH']

# directory for saving results of pCO2SYS calculation for member
esm_output_dir = 'gs://leap-persistent/{INSERT_USERNAME}/phytoplankton/gridded_data_1850-2100'

In [ ]:
def carbonate_system_calculation(path_to_member, output_save_dir, output_vars):
    """
    Calculates and saves carbonate system for ESM output. Requires 'spco2', 'tos', 'sos', 'phos'.
    In order, these are surface ocean (SO) pco2, SO temperature, SO salinity, SO pH, and "Primary Organic Carbon Production by All Types of Phytoplankton".
    This code can take a while to finish running.
    
    Arguments: 
        path_to_member(str): Path to processed ESM member output with which to perform calculation.
        
        output_vars(list): Variables you want to save that are calculated by PyCO2SYS. 
    """
    ### Get scenario, model, member info from path ###
    file_info = path_to_member.split('/')
    scenario = file_info[4]
    assert 'ssp' in scenario
    esm_name = file_info[5]
    member_name = file_info[6]
    member_id = member_name.split('_')[1]

    ##############################################

    ### Setting up saving ###
    output_save_dir = f'{output_save_dir}/{scenario}/{esm_name}/{member_name}'
    output_save_path = f'{output_save_dir}/{esm_name}.{member_id}.carbonate_system_1850-2100.zarr'

    ##############################################

    ### Calculation: ###
    print(output_save_path)
    
    if fs.exists(output_save_path) == False: # only runs if member not used yet
    
        print(f'Getting ESM output from {path_to_member}')
        
        # Open data, truncate to time period
        member_xr = xr.open_dataset('gs://'+path_to_member,engine='zarr')
        member_xr_trunc = member_xr.sel(time=slice('1850','2100'))
        
        # Convert spco2 data to microatmospheres
        member_xr_trunc['spco2'] = member_xr_trunc.spco2/0.101325
        member_xr_trunc['spco2']= member_xr_trunc.spco2.assign_attrs({'units':'microatm'})
        
        # Set up dictionary for chunking
        output_arrays = {var: [] for var in (output_vars or pyco2.sys_output_keys)}
        output_chunks = {}
        
        # Calculate carbonate system for each year (chunks of 12 months)
        # Each 12-month chunk calculation output is saved in an xarray data array.
        # After cycling through each year, the data arrays are all concatenated so they're one file with a continuous time period of 1988-2100.
        # Used chatgpt for time-chunking aspect of code so as to lower memory usage.
        
        for i in range(0, member_xr_trunc.dims['time'], 12):
            print(f'on months {i}-{i+12}/{member_xr_trunc.dims['time']}')
            t_slice = slice(i, i + 12)
            ds_chunk = member_xr_trunc.isel({'time': t_slice})
        
            result = pyco2.sys(par1=ds_chunk.spco2.values, par2=ds_chunk.phos.values, 
                           par1_type=4, par2_type=3,temperature=ds_chunk.tos.values, salinity=ds_chunk.sos.values)
        
            result = dict((k, result[k]) for k in output_vars if k in result)
        
            for var, val in result.items():
                val_da = xr.DataArray(val, dims=('time', 'latitude', 'longitude'),
                                          coords={'time': ds_chunk['time'],
                                                  'latitude': member_xr_trunc['latitude'],
                                                  'longitude': member_xr_trunc['longitude']})
                output_chunks.setdefault(var, []).append(val_da)

        data_vars = {}
        
        # Concatenate all chunks along time
        output_ds = xr.Dataset({
            var: xr.concat(chunks, dim='time') for var, chunks in output_chunks.items()})

        output_ds = output_ds.drop_encoding()
        print(f'Saving result to {output_save_path}')
        output_ds.to_zarr(output_save_path,zarr_format=2)
        print('Saved!')

    elif fs.exists(output_save_path) == True:
        print(f'skipping {path_to_member}, already done')

    ##############################################

In [ ]:
for member_path in fs.glob(f'{esm_output_dir}/*/*/*/*.zarr'):
    # print(member_path)
    carbonate_system_calculation(member_path, esm_output_dir, output_vars)